# Test Notebook: Load PT, POST, PT-G Analyzers

Canonical config: `wd=0.15, bs=1024, ms=1234, ss=42, sh=44`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from analysis.analyzer import ModelAnalyzer, load_model

g++ (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.



In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Model paths (canonical config: wd=0.15, bs=1024, ms=1234, ss=42, sh=44)
pt_path  = PROJECT_ROOT / "outputs/runs/pt/pt_wd0.15_bs1024_ms1234_ss42_sh44_16927810/model.pt"
sft_path = PROJECT_ROOT / "outputs/runs/sft/sft_wd0.15_bs1024_ms1234_ss42_sh44_16929321/model.pt"
ptg_path = PROJECT_ROOT / "outputs/runs/pt-g/ptg_wd0.15_bs1024_ms1234_ss42_sh44_16926075/model.pt"

for p in [pt_path, sft_path, ptg_path]:
    assert p.exists(), f"Missing: {p}"
print("All model files found.")

Device: cuda
All model files found.


In [3]:
# Load models
pt_model  = load_model(pt_path)
sft_model = load_model(sft_path)
ptg_model = load_model(ptg_path)
print("Models loaded.")

Models loaded.


In [4]:
# Create analyzers
a_pt  = ModelAnalyzer(pt_model,  task="pt",  device=device, label="PT")
a_sft = ModelAnalyzer(sft_model, task="ptg", device=device, label="POST")
a_ptg = ModelAnalyzer(ptg_model, task="ptg", device=device, label="PT-G")
print("Analyzers ready.")

Moving model to device:  cuda
Moving model to device:  cuda
Moving model to device:  cuda
Moving model to device:  cuda
Moving model to device:  cuda
Moving model to device:  cuda
Analyzers ready.


In [5]:
# Quick sanity check: evaluate all three
eval_pt  = a_pt.evaluate(train_frac=0.3, seed=42)
eval_sft = a_sft.evaluate(train_frac=0.3, seed=42)
eval_ptg = a_ptg.evaluate(train_frac=0.3, seed=42)

for name, e in [("PT", eval_pt), ("POST", eval_sft), ("PT-G", eval_ptg)]:
    print(f"{name:5s}  test_loss={e['test_loss']:.2e}  test_acc={e['test_acc']:.4f}")

PT     test_loss=1.84e-07  test_acc=1.0000
POST   test_loss=2.08e-07  test_acc=1.0000
PT-G   test_loss=6.82e-07  test_acc=1.0000


In [16]:
# eq_logits: logits at the "=" position, shape (p, p, vocab)
# eq_logits[a, b] = logit vector when input is (a + b)
eq_logits_pt  = a_pt.eq_logits
eq_logits_sft = a_sft.eq_logits
eq_logits_ptg = a_ptg.eq_logits

print(f"eq_logits shape: {eq_logits_pt.shape}")  # (113, 113, vocab)

eq_logits shape: torch.Size([113, 113, 117])


In [17]:
eq_logits_sft.mean((0,1))

tensor([-144.5813,  -98.1832, -139.9547, -105.2359, -150.8811, -100.6819,
        -147.2628, -101.2266, -148.7142, -104.1030, -150.5495, -102.7574,
        -148.7424, -106.2754, -153.1023,  -99.2739, -139.1266,  -92.6578,
        -162.0941,  -96.5115, -140.8802, -106.6420, -146.8088,  -98.9285,
        -137.9881, -100.9548, -139.0547, -104.5249, -128.4386, -103.9718,
        -166.6097, -102.3584, -170.2947,  -97.8488, -134.6330, -101.8819,
        -171.4442,  -98.0481, -162.5745, -104.7473, -157.9217,  -98.9534,
        -170.3845, -105.2481, -143.9832, -104.4499, -134.6587,  -97.0146,
        -156.6855, -105.2056, -146.5671,  -98.6766, -138.4968, -101.2821,
        -163.6552,  -99.6532, -156.0056, -105.5999, -153.0158,  -99.1889,
        -178.6896,  -99.4282, -176.9585, -100.8863, -147.6530, -103.7858,
        -140.7352, -101.1701, -144.2968, -103.7348, -146.9485, -103.4838,
        -142.3594, -100.0190, -166.2859,  -99.4954, -132.8780,  -99.8308,
        -147.2984, -104.9471, -182.931

In [15]:
eq_logits_sft[0,2]

tensor([ -64.2076,   35.2001,   74.5853,   31.5637,  -63.5238,   33.3132,
          38.0868,   29.1130,  -55.9253,   20.2241,  -40.3335,   26.3053,
         -25.0562,   12.1787, -101.8659,   26.2547,   16.3817,   34.6573,
        -122.2539,   33.5647,   56.0569,   23.8255,  -63.2066,   37.0844,
          37.6331,   33.6055,   -9.6036,   24.6979,   -2.1597,   32.0582,
           8.5699,   20.8945, -128.9645,   33.1047,   27.7088,   16.9724,
        -154.9949,   29.6978,    6.6314,   17.9981,  -82.4054,   28.8087,
         -21.9293,   25.5490,    0.3304,   25.0488,  -26.1782,   42.7736,
          51.6988,   23.7103,  -74.6530,   42.1113,   47.5928,   24.5856,
        -112.3811,   33.6592,   -5.4834,   16.0175,  -59.7654,   22.2785,
         -88.7099,   29.8584,  -29.6449,   17.2743,  -79.5555,   35.2547,
          45.4599,   22.8507,  -67.0986,   44.7780,   53.9065,   25.4250,
         -45.6524,   38.1905,   -1.5222,   29.1119,   -2.3596,   21.0876,
         -64.5598,   28.1973,  -39.861